# 05 · Zulu translation (RQ2)

Translates the 2,356 sampled prompts into Zulu with NLLB-200 (distilled 1.3B), sentence by sentence, keeping line breaks and leaving pieces without letters unchanged.

Input: `stage1_prompts.csv` (notebook 02). Output: `zulu_full.csv`.

In [ ]:
import os, re
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import drive

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/jailbreak_project"
os.makedirs(f"{SAVE_DIR}/rq2", exist_ok=True)
OUT = f"{SAVE_DIR}/rq2/zulu_full.csv"

prompts = pd.read_csv(f"{SAVE_DIR}/relabel_api/stage1_prompts.csv", keep_default_na=False, dtype={"prompt": str})
done = set(pd.read_csv(OUT, usecols=["uid"]).uid) if os.path.exists(OUT) else set()
todo = prompts[~prompts.uid.isin(done)]

## 1. Splitting

In [ ]:
SENT_SPLIT = re.compile(r"(?<=[.!?])(\s+)")
HAS_LETTER = re.compile(r"[^\W\d_]")
MAX_WORDS, MAX_CHARS = 120, 700


def plan(text):
    """Split a prompt into ("tr", piece) items to translate and ("keep", piece) items kept as they are
    (line breaks, spacing, pieces without letters)."""
    items = []
    for li, line in enumerate(text.split("\n")):
        if li > 0:
            items.append(("keep", "\n"))
        core = line.strip()
        if not core:
            if line:
                items.append(("keep", line))
            continue
        lead, trail = line[:len(line) - len(line.lstrip())], line[len(line.rstrip()):]
        if lead:
            items.append(("keep", lead))
        for pi, part in enumerate(SENT_SPLIT.split(core)):      # sentence, separator, sentence, ...
            if pi % 2 == 1 or not HAS_LETTER.search(part):
                items.append(("keep", part))
                continue
            words = part.split(" ")
            for k in range(0, len(words), MAX_WORDS):            # long sentences: at most 120 words per piece
                if k > 0:
                    items.append(("keep", " "))
                items += cut(" ".join(words[k:k + MAX_WORDS]))
        if trail:
            items.append(("keep", trail))
    return items


def cut(piece):
    """Cut a piece longer than 700 characters, at a space when possible."""
    def item(s):
        return ("tr", s) if HAS_LETTER.search(s) else ("keep", s)
    out = []
    while len(piece) > MAX_CHARS:
        at = piece.rfind(" ", 0, MAX_CHARS)
        if at > 0:
            out += [item(piece[:at]), ("keep", " ")]
            piece = piece[at + 1:]
        else:
            out.append(item(piece[:MAX_CHARS]))
            piece = piece[MAX_CHARS:]
    if piece:
        out.append(item(piece))
    return out


def assemble(items, translated):
    it = iter(translated)
    return "".join(next(it) if kind == "tr" else s for kind, s in items)

## 2. NLLB-200

Beam search with 4 beams and no sampling, so the translations are deterministic.

In [ ]:
MODEL = "facebook/nllb-200-distilled-1.3B"
tok = AutoTokenizer.from_pretrained(MODEL, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, torch_dtype=torch.float16).to("cuda").eval()
ZULU_ID = tok.convert_tokens_to_ids("zul_Latn")


@torch.inference_mode()
def translate(pieces, batch_size=32):
    out = [None] * len(pieces)
    order = sorted(range(len(pieces)), key=lambda i: len(pieces[i]))   # similar lengths in each batch
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        enc = tok([pieces[i] for i in idx], return_tensors="pt", padding=True, truncation=True,
                  max_length=512).to("cuda")
        gen = model.generate(**enc, forced_bos_token_id=ZULU_ID, num_beams=4,
                             max_new_tokens=min(512, int(enc["input_ids"].shape[1] * 1.6) + 20))
        for i, text in zip(idx, tok.batch_decode(gen, skip_special_tokens=True)):
            out[i] = text
    return out

## 3. Translate

Prompts are translated in groups of 20 and appended to `zulu_full.csv`.

In [ ]:
for g in range(0, len(todo), 20):
    group = todo.iloc[g:g + 20]
    plans = [plan(p) for p in group.prompt]
    zulu = translate([piece for pl in plans for kind, piece in pl if kind == "tr"])
    rows, pos = [], 0
    for (_, r), pl in zip(group.iterrows(), plans):
        n = sum(kind == "tr" for kind, _ in pl)
        zu = assemble(pl, zulu[pos:pos + n])
        pos += n
        rows.append({"uid": r.uid, "zu": zu})
    pd.DataFrame(rows).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)